# Khởi tạo dữ liệu

In [2]:
import sqlite3
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Tạo CSDL trong bộ nhớ
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Tạo bảng student
cur.execute("""
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

# Tạo bảng course
cur.execute("""
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
""")

# Chèn dữ liệu student
students = [
    (1,  "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2,  "Tran Thi Lan",      "Kinh Te", 34, 9.2),
    (3,  "Pham Van Nam",      "Toan Tin", None, 7.9),
    (4,  "Le Thanh Huyen",    "Toan Tin", 20, 7.2),
    (5,  "Vu Quoc Anh",       "May Tinh", 24, 8.0),
    (6,  "Dang Thuy Linh",    "May Tinh", 24, 5.5),
    (7,  "Bui Tien Dung",     "Kinh Te", 34, 9.2),
    (8,  "Ho Ngoc Mai",       "Toan Tin", 20, 8.8),
    (9,  "Duong Huu Phuc",    "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh",      "May Tinh", None, 7.0)
]

# Chèn dữ liệu course
courses = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

cur.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)
cur.executemany("INSERT INTO course VALUES (?, ?)", courses)
conn.commit()

# Hiển thị dữ liệu ban đầu
print("Bảng student")
display(pd.read_sql_query("SELECT * FROM student", conn))

print("Bảng course")
display(pd.read_sql_query("SELECT * FROM course", conn))

Bảng student


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


Bảng course


,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


# Câu 1

In [3]:
# 1. Tích Descartes
df_cartesian = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_table_id, c.course_name
FROM student s
CROSS JOIN course c
ORDER BY s.student_id, c.id
""", conn)

print("1. Tích Descartes")
print("Số dòng:", len(df_cartesian))
display(df_cartesian)

# 2. INNER JOIN
df_inner = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
INNER JOIN course c
    ON s.course_id = c.id
ORDER BY s.student_id
""", conn)

print("2. INNER JOIN")
print("Số dòng:", len(df_inner))
display(df_inner)

# 3. LEFT JOIN
df_left = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id
ORDER BY s.student_id
""", conn)

print("3. LEFT JOIN")
print("Số dòng:", len(df_left))
display(df_left)

# 4. RIGHT JOIN mô phỏng
df_right = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id AS course_table_id, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
ORDER BY c.id, s.student_id
""", conn)

print("4. RIGHT JOIN (mô phỏng trong SQLite)")
print("Số dòng:", len(df_right))
display(df_right)

# 5. FULL OUTER JOIN mô phỏng
df_full = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_table_id, c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id

UNION ALL

SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_table_id, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
WHERE s.student_id IS NULL
ORDER BY student_id, course_table_id
""", conn)

print("5. FULL OUTER JOIN (mô phỏng trong SQLite)")
print("Số dòng:", len(df_full))
display(df_full)

1. Tích Descartes
Số dòng: 30


,student_id,name,class,course_id,score,course_table_id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


2. INNER JOIN
Số dòng: 3


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


3. LEFT JOIN
Số dòng: 10


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


4. RIGHT JOIN (mô phỏng trong SQLite)
Số dòng: 4


,student_id,name,class,course_id,score,course_table_id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,None,None,NaN,NaN,26,Tin hoc
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
3,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke


5. FULL OUTER JOIN (mô phỏng trong SQLite)
Số dòng: 11


,student_id,name,class,course_id,score,course_table_id,course_name
0,NaN,None,None,NaN,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None


# Câu 2

In [5]:
# Cập nhật course_id còn thiếu theo giả định hợp lý
cur.execute("""
UPDATE student
SET course_id = CASE
    WHEN class = 'Kinh Te' THEN 34
    WHEN class = 'May Tinh' THEN 26
    WHEN class = 'Toan Tin' THEN 12
    ELSE course_id
END
WHERE course_id IS NULL
""")
conn.commit()

print("Sau khi cập nhật course_id bị thiếu")
display(pd.read_sql_query("SELECT * FROM student ORDER BY student_id", conn))

# Xóa các bản ghi có course_id không tồn tại trong course
cur.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

print("Sau khi xóa các bản ghi có course_id không tồn tại trong course")
display(pd.read_sql_query("SELECT * FROM student ORDER BY student_id", conn))

Sau khi cập nhật course_id bị thiếu


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,12,7.9
3,4,Le Thanh Huyen,Toan Tin,20,7.2
4,5,Vu Quoc Anh,May Tinh,24,8.0
5,6,Dang Thuy Linh,May Tinh,24,5.5
6,7,Bui Tien Dung,Kinh Te,34,9.2
7,8,Ho Ngoc Mai,Toan Tin,20,8.8
8,9,Duong Huu Phuc,Kinh Te,34,7.2
9,10,Cao Thi Hanh,May Tinh,26,7.0


Sau khi xóa các bản ghi có course_id không tồn tại trong course


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,12,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,34,7.2
5,10,Cao Thi Hanh,May Tinh,26,7.0


## Câu 2a

In [6]:
df_class_stats = pd.read_sql_query("""
SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class
ORDER BY class
""", conn)

print("Tổng số sinh viên và điểm trung bình của từng lớp")
display(df_class_stats)

Tổng số sinh viên và điểm trung bình của từng lớp


,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


## Câu 2b

In [7]:
df_course_stats = pd.read_sql_query("""
SELECT c.id AS course_id,
       c.course_name,
       COUNT(*) AS total_students,
       ROUND(AVG(s.score), 2) AS avg_score
FROM student s
JOIN course c
    ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY c.id
""", conn)

print("Tổng số sinh viên và điểm trung bình của từng môn học")
display(df_course_stats)

Tổng số sinh viên và điểm trung bình của từng môn học


,course_id,course_name,total_students,avg_score
0,12,Giai tich,2,7.30
1,26,Tin hoc,1,7.00
2,34,Thong ke,3,8.53


## Câu 2c

In [8]:
df_course_classification = pd.read_sql_query("""
SELECT c.id AS course_id,
       c.course_name,
       ROUND(AVG(s.score), 2) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6.0 THEN 'Tot'
           ELSE 'Kem'
       END AS classification
FROM student s
JOIN course c
    ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY c.id
""", conn)

print("Phân loại môn học theo điểm trung bình")
display(df_course_classification)

Phân loại môn học theo điểm trung bình


,course_id,course_name,avg_score,classification
0,12,Giai tich,7.30,Tot
1,26,Tin hoc,7.00,Tot
2,34,Thong ke,8.53,Tot


# Câu 3


In [9]:
df_rank_all = pd.read_sql_query("""
SELECT student_id, name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
ORDER BY rank_score, student_id
""", conn)

print("Xếp hạng sinh viên theo điểm số toàn bộ")
display(df_rank_all)

print("Top 3 sinh viên điểm cao nhất")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score DESC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
ORDER BY rank_score, student_id
""", conn))

print("Top 3 sinh viên điểm thấp nhất")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score ASC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
ORDER BY rank_score, student_id
""", conn))

Xếp hạng sinh viên theo điểm số toàn bộ


,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3
3,9,Duong Huu Phuc,Kinh Te,34,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


Top 3 sinh viên điểm cao nhất


,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3


Top 3 sinh viên điểm thấp nhất


,student_id,name,class,course_id,score,rank_score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3


In [10]:
df_rank_class = pd.read_sql_query("""
SELECT student_id, name, class, course_id, score,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student
ORDER BY class, rank_in_class, student_id
""", conn)

print("Xếp hạng sinh viên theo lớp")
display(df_rank_class)

print("Top 3 cao nhất theo từng lớp")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
ORDER BY class, rank_in_class, student_id
""", conn))

print("Top 3 thấp nhất theo từng lớp")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
ORDER BY class, rank_in_class, student_id
""", conn))

Xếp hạng sinh viên theo lớp


,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,12,7.9,1


Top 3 cao nhất theo từng lớp


,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,12,7.9,1


Top 3 thấp nhất theo từng lớp


,student_id,name,class,course_id,score,rank_in_class
0,9,Duong Huu Phuc,Kinh Te,34,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,2
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
4,10,Cao Thi Hanh,May Tinh,26,7.0,2
5,3,Pham Van Nam,Toan Tin,12,7.9,1


In [12]:
df_rank_course = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
       RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_in_course
FROM student s
JOIN course c
    ON s.course_id = c.id
ORDER BY s.course_id, rank_in_course, s.student_id
""", conn)

print("Xếp hạng sinh viên theo mã môn học")
display(df_rank_course)

print("Top 3 cao nhất theo từng môn học")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_in_course
    FROM student s
    JOIN course c
        ON s.course_id = c.id
)
WHERE rank_in_course <= 3
ORDER BY course_id, rank_in_course, student_id
""", conn))

print("Top 3 thấp nhất theo từng môn học")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score ASC) AS rank_in_course
    FROM student s
    JOIN course c
        ON s.course_id = c.id
)
WHERE rank_in_course <= 3
ORDER BY course_id, rank_in_course, student_id
""", conn))

Xếp hạng sinh viên theo mã môn học


,student_id,name,class,course_id,course_name,score,rank_in_course
0,3,Pham Van Nam,Toan Tin,12,Giai tich,7.9,1
1,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,2
2,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0,1
3,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,3


Top 3 cao nhất theo từng môn học


,student_id,name,class,course_id,course_name,score,rank_in_course
0,3,Pham Van Nam,Toan Tin,12,Giai tich,7.9,1
1,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,2
2,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0,1
3,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,3


Top 3 thấp nhất theo từng môn học


,student_id,name,class,course_id,course_name,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,1
1,3,Pham Van Nam,Toan Tin,12,Giai tich,7.9,2
2,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0,1
3,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,1
4,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,2
5,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,2


# Câu 4


In [13]:
# Thêm cột graduation_date
cur.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
conn.commit()

# Cập nhật graduation_date = thời gian hiện tại + số hạng tháng theo điểm số
cur.execute("""
WITH ranked AS (
    SELECT student_id,
           RANK() OVER (ORDER BY score DESC) AS rnk
    FROM student
)
UPDATE student
SET graduation_date = DATETIME(
    'now',
    '+' || (SELECT rnk FROM ranked WHERE ranked.student_id = student.student_id) || ' months'
)
""")
conn.commit()

df_graduation = pd.read_sql_query("""
SELECT student_id, name, class, course_id, score, graduation_date
FROM student
ORDER BY student_id
""", conn)

print("Bảng student sau khi thêm graduation_date")
display(df_graduation)

Bảng student sau khi thêm graduation_date


,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-10-03 09:53:45
1,2,Tran Thi Lan,Kinh Te,34,9.2,2026-05-03 09:53:45
2,3,Pham Van Nam,Toan Tin,12,7.9,2026-07-03 09:53:45
3,7,Bui Tien Dung,Kinh Te,34,9.2,2026-05-03 09:53:45
4,9,Duong Huu Phuc,Kinh Te,34,7.2,2026-08-03 09:53:45
5,10,Cao Thi Hanh,May Tinh,26,7.0,2026-09-03 09:53:45
